<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright (c) 2025 FlyDSL Project Contributors -->

# Target-agnostic operations: the `Universal*` atoms

FlyDSL expresses data movement and math through **atoms** — small, named
operations like "copy 32 bits" or "multiply-accumulate". Atoms come in two flavors:

- **Target-agnostic** `Universal*` atoms describe *what* to do. The backend decides
  *how* to realize it on the actual GPU.
- **Architecture-specific** atoms under `fx.rocdl.*` (e.g. `BufferCopy32b`, `MFMA`)
  name a particular hardware instruction family (here, AMD CDNA buffer ops / MFMA).

This notebook focuses on the universal flavor — write once, let the compiler
specialize — and ends with a complete vector-add kernel built entirely from
universal atoms.

In [ ]:
import torch

import flydsl.compiler as flyc
import flydsl.expr as fx

## 1. The universal atom family

- **Copy:** `UniversalCopy8b`, `UniversalCopy16b`, `UniversalCopy32b`,
  `UniversalCopy64b`, `UniversalCopy128b` — move *N* bits between tensors.
- **Multiply-accumulate:** `UniversalFMA` — the target-agnostic MMA atom.
- **Atomics:** `UniversalAtomicAdd`, `UniversalAtomicMax`, `UniversalAtomicMin`,
  `UniversalAtomicAnd/Or`, `UniversalAtomicInc/Dec`.

We use the **copy** atoms here, since they need nothing but a tensor and a dtype.
`UniversalFMA` and the atomics belong to the same family but come into their own
alongside tiled layouts and MMA, which are covered in a later series.

You build a copy atom and invoke it like this:

```python
atom = fx.make_copy_atom(fx.UniversalCopy32b(), fx.Float32)
fx.copy_atom_call(atom, src, dst)   # copy src -> dst
```

Swapping `fx.UniversalCopy32b()` for `fx.rocdl.BufferCopy32b()` would pin this to
CDNA buffer loads/stores; the universal atom keeps it portable.

## 2. Capstone: a vector add, fully universal

Each thread loads one element of `A` and one of `B` into registers, adds them, and
stores the result to `C`. The loads and the store are all `UniversalCopy32b`.

We do use a little layout machinery (`make_layout`, `logical_divide`, `slice`)
purely to hand each thread *its* element — that is the subject of the next series,
so don't dwell on it here. The point of this notebook is the **copy**: the exact
same `copy_atom_call` code runs on any FlyDSL target.

In [ ]:
# Turn IR dumping on *before* the first compile, so we can inspect it in the next cell.
import contextlib
import glob
import io
import os
import tempfile

dump_dir = tempfile.mkdtemp(prefix="flydsl_ir_")
os.environ["FLYDSL_DUMP_IR"] = "1"
os.environ["FLYDSL_DUMP_DIR"] = dump_dir


@flyc.kernel
def vadd_kernel(A: fx.Tensor, B: fx.Tensor, C: fx.Tensor, block_dim: fx.Constexpr[int]):
    bid = fx.block_idx.x
    tid = fx.thread_idx.x

    # Hand thread (bid, tid) its single element of each vector. This little
    # partitioning is layout algebra — the topic of the next series; don't dwell on it.
    tA = fx.logical_divide(A, fx.make_layout(block_dim, 1))
    tB = fx.logical_divide(B, fx.make_layout(block_dim, 1))
    tC = fx.logical_divide(C, fx.make_layout(block_dim, 1))
    tA = fx.logical_divide(fx.slice(tA, (None, bid)), fx.make_layout(1, 1))
    tB = fx.logical_divide(fx.slice(tB, (None, bid)), fx.make_layout(1, 1))
    tC = fx.logical_divide(fx.slice(tC, (None, bid)), fx.make_layout(1, 1))

    # the target-agnostic copy atom
    copy = fx.make_copy_atom(fx.UniversalCopy32b(), fx.Float32)

    rA = fx.make_rmem_tensor(fx.make_layout(1, 1), fx.Float32)
    rB = fx.make_rmem_tensor(fx.make_layout(1, 1), fx.Float32)
    rC = fx.make_rmem_tensor(fx.make_layout(1, 1), fx.Float32)

    fx.copy_atom_call(copy, fx.slice(tA, (None, tid)), rA)   # global -> register
    fx.copy_atom_call(copy, fx.slice(tB, (None, tid)), rB)

    vC = fx.arith.addf(fx.memref_load_vec(rA), fx.memref_load_vec(rB))
    fx.memref_store_vec(vC, rC)

    fx.copy_atom_call(copy, rC, fx.slice(tC, (None, tid)))    # register -> global


@flyc.jit
def vadd(A: fx.Tensor, B: fx.Tensor, C, n: fx.Int32, stream: fx.Stream = fx.Stream(None)):
    block_dim = 64
    grid_x = (n + block_dim - 1) // block_dim
    vadd_kernel(A, B, C, block_dim).launch(grid=(grid_x, 1, 1), block=[block_dim, 1, 1], stream=stream)


# Run it and validate against torch (hiding the verbose per-pass dump log).
n = 256
A = torch.randint(0, 10, (n,), dtype=torch.float32).cuda()
B = torch.randint(0, 10, (n,), dtype=torch.float32).cuda()
C = torch.zeros(n, dtype=torch.float32).cuda()
tA = flyc.from_dlpack(A).mark_layout_dynamic(leading_dim=0, divisibility=4)

with contextlib.redirect_stdout(io.StringIO()):
    vadd(tA, B, C, n, stream=torch.cuda.Stream())
    torch.cuda.synchronize()

os.environ.pop("FLYDSL_DUMP_IR", None)  # stop dumping for any later compiles
print("matches torch:", torch.allclose(C, A + B))
print("C[:8] =", C[:8].tolist())

## 3. Seeing "universal" in the IR

Dump the IR and look at the high-level `00_origin.mlir`: the copies appear as
generic `fly` copy-atom ops, with no architecture baked in. The specialization to
this gfx950 box happens later, in the `convert_fly_to_rocdl` pass.

In [ ]:
origin = sorted(glob.glob(os.path.join(dump_dir, "*", "00_origin.mlir")))[0]
atom_lines = [ln.strip() for ln in open(origin).read().splitlines() if "copy_atom" in ln]
print("copy ops in 00_origin.mlir — note the target-agnostic `universal_copy<32>`:\n")
print("\n".join(atom_lines))

## Recap

Across these four notebooks you have the `flydsl.expr` foundation:

- **00** — the `@kernel` / `@jit` trace model and how to read dumped IR.
- **01** — the numeric type system: ints, floats, `bf16`/`fp8`, casts, `Constexpr`.
- **02** — `@fx.struct` aggregate value types and their memory layout.
- **03** — target-agnostic `Universal*` atoms and a complete vector add.

**Next series:** layout algebra — `make_layout`, `logical_divide`, `partition`,
tiled copy, and the MMA atoms (`UniversalFMA` / `rocdl.MFMA`). That is where the
small slicing we glossed over above is explained from first principles.